# ETL — Minnie65: Cell-Cell Connectivity (Long)

Writes `CellCellConnectivityLong` synapse data for Minnie65 v1300 using two examples from the same precomputed parquet:

1. **Example 1** — (proofread ∩ CSM)-pre × CSM-post, two measurement types (`SYNAPSE_COUNT`, `SUM_ANATOMICAL_SIZE`).
2. **Example 2** — (proofread ∩ CSM)-pre × (proofread ∩ CSM)-post, `SYNAPSE_COUNT` only.

Also creates the `minnie65_v1300_proofread` cohort DataSet (cells that are both proofread AND in the CSM cohort) and its associations.

**Outputs:**
- `dataset/` — +1 row (`minnie65_v1300_proofread`)
- `dataitem_dataset_association/` — one per proofread ∩ CSM cell
- `cellcellconnectivitylong_proofread_pre_to_csm_post/` — 2 rows per pair
- `cellcellconnectivitylong_proofread_to_proofread/` — 1 row per pair

**Prerequisites:** `etl_minnie_01_dataset_dataitem.ipynb`, `etl_minnie_02_cell_features.ipynb`.

Identifiers: `project_id = "minnie65"`, `dataset_id = "minnie65_v1300_proofread"`.

In [14]:
import os

import caveclient
import pandas as pd
import polars as pl
import pyarrow as pa
from deltalake import write_deltalake

from connects_common_connectivity.io.arrow_utils import (
    build_arrow_schema,
    models_to_table,
    attach_linkml_metadata,
)
from connects_common_connectivity.models import (
    CellCellConnectivityLong,
    DataSet,
    DataItemDataSetAssociation,
    Modality,
    SynapticMeasurementType,
    Unit,
)

from connects_common_connectivity.config import output_root
from connects_common_connectivity.io import write_models

In [15]:
OUTPUT_ROOT = output_root()
PROJECT_ID = "minnie65"
DATASET_ID_CSM = "minnie65_v1300_csm_cluster"
DATASET_ID_PROOFREAD = "minnie65_v1300_proofread"
PARQUET_PATH = "/data/minnie1412/minnie_soma_soma_connectivity.parquet"
CAVE_DATASTACK = "minnie65_phase3_v1"
CAVE_VERSION = 1300
CAVE_NUC_VIEW = "nucleus_detection_lookup_v1"
CAVE_PROOFREAD_TABLE = "proofreading_status_and_strategy"

print(f"OUTPUT_ROOT            : {OUTPUT_ROOT}")
print(f"PROJECT_ID             : {PROJECT_ID}")
print(f"DATASET_ID_CSM         : {DATASET_ID_CSM}")
print(f"DATASET_ID_PROOFREAD   : {DATASET_ID_PROOFREAD}")
print(f"PARQUET_PATH           : {PARQUET_PATH}")
print(f"CAVE_DATASTACK         : {CAVE_DATASTACK}")
print(f"CAVE_VERSION           : {CAVE_VERSION}")
print(f"CAVE_NUC_VIEW          : {CAVE_NUC_VIEW}")
print(f"CAVE_PROOFREAD_TABLE   : {CAVE_PROOFREAD_TABLE}")

OUTPUT_ROOT            : ../scratch/em_patchseq_wnm_v2/
PROJECT_ID             : minnie65
DATASET_ID_CSM         : minnie65_v1300_csm_cluster
DATASET_ID_PROOFREAD   : minnie65_v1300_proofread
PARQUET_PATH           : /data/minnie1412/minnie_soma_soma_connectivity.parquet
CAVE_DATASTACK         : minnie65_phase3_v1
CAVE_VERSION           : 1300
CAVE_NUC_VIEW          : nucleus_detection_lookup_v1
CAVE_PROOFREAD_TABLE   : proofreading_status_and_strategy


In [16]:
# Prereq: CSM cohort associations must exist (written by etl_minnie_02)
csm_assoc = pl.read_delta(OUTPUT_ROOT + "dataitem_dataset_association/").filter(
    (pl.col("project_id") == PROJECT_ID) & (pl.col("dataset_id") == DATASET_ID_CSM)
)
assert csm_assoc.shape[0] > 0, (
    "No CSM cohort associations found. Run etl_minnie_02_cell_features.ipynb first."
)
csm_nuc_ids = set(csm_assoc["dataitem_id"].to_list())
print(f"CSM cohort cells loaded: {len(csm_nuc_ids)}")

CSM cohort cells loaded: 35783


---
## Build proofread ∩ CSM cohort

Query CAVE `proofreading_status_and_strategy` table (filter `status_axon`), join to `nucleus_detection_lookup_v1` to get nucleus ids, then intersect with the CSM cohort. The resulting set defines the `minnie65_v1300_proofread` DataSet.

In [17]:
client = caveclient.CAVEclient(CAVE_DATASTACK, auth_token=os.environ["CUSTOM_KEY"])
client.materialize.version = CAVE_VERSION

# Get proofread cells (status_axon == True)
proof_df = client.materialize.query_table(
    CAVE_PROOFREAD_TABLE, materialization_version=CAVE_VERSION
)
proof_df = proof_df.query("status_axon == True")
print(f"Proofread cells (status_axon): {proof_df.shape[0]}")

# Get nucleus lookup to map pt_root_id → nucleus id
nuc_df = client.materialize.query_view(CAVE_NUC_VIEW)
nuc_df = nuc_df.query("pt_root_id != 0")
print(f"Nucleus lookup rows: {nuc_df.shape[0]}")

# Join: proofread pt_root_id → nucleus id
proof_with_nuc = proof_df.merge(
    nuc_df[["id", "pt_root_id"]].rename(columns={"id": "nuc_id"}),
    on="pt_root_id",
    how="inner",
)
all_proofread_nuc_ids = set(proof_with_nuc["nuc_id"].astype(str).tolist())
print(f"Proofread cells with nucleus id: {len(all_proofread_nuc_ids)}")

Proofread cells (status_axon): 1723
Nucleus lookup rows: 133969
Proofread cells with nucleus id: 1711


In [18]:
# The proofread cohort DataSet is defined as proofread ∩ CSM
proofread_nuc_ids = all_proofread_nuc_ids & csm_nuc_ids
proof_only = all_proofread_nuc_ids - csm_nuc_ids

print(f"|all proofread|        : {len(all_proofread_nuc_ids)}")
print(f"|csm|                  : {len(csm_nuc_ids)}")
print(f"|proofread ∩ csm|      : {len(proofread_nuc_ids)}  ← this is the proofread DataSet")
print(f"|proofread \\ csm|      : {len(proof_only)}")

|all proofread|        : 1711
|csm|                  : 35783
|proofread ∩ csm|      : 1636  ← this is the proofread DataSet
|proofread \ csm|      : 75


The `minnie65_v1300_proofread` DataSet contains cells that have both verified axon reconstructions AND CSM dendrite classifications. These cells form the pre-synaptic population for both examples. Example 1 uses all CSM cells as the post-synaptic population; Example 2 restricts both sides to the proofread ∩ CSM set.

In [19]:
# All proofread ∩ CSM cells must already be DataItems (registered by etl_minnie_01)
existing_items = set(
    pl.read_delta(OUTPUT_ROOT + "dataitem/")
    .filter(pl.col("project_id") == PROJECT_ID)["id"]
    .to_list()
)
missing = proofread_nuc_ids - existing_items
assert len(missing) == 0, (
    f"{len(missing)} proofread ∩ CSM nucleus ids not found in dataitem/. "
    "Run etl_minnie_01_dataset_dataitem.ipynb first."
)
print(f"All {len(proofread_nuc_ids)} proofread ∩ CSM cells confirmed in dataitem/.")

All 1636 proofread ∩ CSM cells confirmed in dataitem/.


In [20]:
proofread_ds = DataSet(
    id=DATASET_ID_PROOFREAD,
    name="Minnie65 v1300 proofread axon cohort",
    modality=Modality.ELECTRON_MICROSCOPY.value,
    project_id=PROJECT_ID,
)

result_ds = write_models(proofread_ds, output_root=OUTPUT_ROOT)
print(f"DataSet written: {result_ds.rows_written} row(s)")

DataSet written: 1 row(s)


In [21]:
ds_v = pl.read_delta(OUTPUT_ROOT + "dataset/").filter(
    (pl.col("project_id") == PROJECT_ID) & (pl.col("id") == DATASET_ID_PROOFREAD)
)
print(ds_v.shape)
print(ds_v.head(3))
assert ds_v.shape[0] == 1

(1, 5)
shape: (1, 5)
┌──────────────────────────┬──────────────────┬─────────────┬─────────────────────┬────────────┐
│ id                       ┆ name             ┆ publication ┆ modality            ┆ project_id │
│ ---                      ┆ ---              ┆ ---         ┆ ---                 ┆ ---        │
│ str                      ┆ str              ┆ str         ┆ str                 ┆ str        │
╞══════════════════════════╪══════════════════╪═════════════╪═════════════════════╪════════════╡
│ minnie65_v1300_proofread ┆ Minnie65 v1300   ┆ null        ┆ ELECTRON_MICROSCOPY ┆ minnie65   │
│                          ┆ proofread axon … ┆             ┆                     ┆            │
└──────────────────────────┴──────────────────┴─────────────┴─────────────────────┴────────────┘


In [22]:
associations = [
    DataItemDataSetAssociation(
        dataitem_id=nuc_id, dataset_id=DATASET_ID_PROOFREAD, project_id=PROJECT_ID
    )
    for nuc_id in sorted(proofread_nuc_ids)
]

result_assoc = write_models(associations, output_root=OUTPUT_ROOT)
print(f"DataItemDataSetAssociation written: {result_assoc.rows_written} row(s) (proofread ∩ CSM)")

DataItemDataSetAssociation written: 1636 row(s) (proofread ∩ CSM)


In [23]:
assoc_v = pl.read_delta(OUTPUT_ROOT + "dataitem_dataset_association/").filter(
    (pl.col("project_id") == PROJECT_ID) & (pl.col("dataset_id") == DATASET_ID_PROOFREAD)
)
print(assoc_v.shape)
print(assoc_v.head(3))
assert assoc_v.shape[0] == len(proofread_nuc_ids)

(1636, 3)
shape: (3, 3)
┌─────────────┬──────────────────────────┬────────────┐
│ dataitem_id ┆ dataset_id               ┆ project_id │
│ ---         ┆ ---                      ┆ ---        │
│ str         ┆ str                      ┆ str        │
╞═════════════╪══════════════════════════╪════════════╡
│ 188569      ┆ minnie65_v1300_proofread ┆ minnie65   │
│ 188961      ┆ minnie65_v1300_proofread ┆ minnie65   │
│ 189149      ┆ minnie65_v1300_proofread ┆ minnie65   │
└─────────────┴──────────────────────────┴────────────┘


---
## Load connectivity parquet

Both examples below use the same precomputed soma-to-soma connectivity parquet, filtered differently.

In [24]:
conn_df = pl.read_parquet(PARQUET_PATH)
print(f"Raw parquet rows: {conn_df.shape[0]}")

# Cast nuc_id columns to string for matching
conn_df = conn_df.with_columns(
    pl.col("pre_nuc_id").cast(pl.Utf8).alias("pre_nuc_id_str"),
    pl.col("post_nuc_id").cast(pl.Utf8).alias("post_nuc_id_str"),
)
conn_df.head(3)

Raw parquet rows: 796098


pre_pt_root_id,post_pt_root_id,n_syn,sum_size,pre_nuc_id,post_nuc_id,__index_level_0__,pre_nuc_id_str,post_nuc_id_str
i64,i64,i64,i64,i64,i64,i64,str,str
864691135136899865,864691134884743162,1,5984,337175,304043,43176584,"""337175""","""304043"""
864691135360346200,864691134884756730,1,5920,330167,339142,43181492,"""330167""","""339142"""
864691135373601736,864691134884756730,1,3852,273595,339142,43181499,"""273595""","""339142"""


---
## Example 1 — (proofread ∩ CSM)-pre × CSM-post

Filter to pre-synaptic cells in the proofread ∩ CSM set and post-synaptic cells in the full CSM set. Write two `CellCellConnectivityLong` measurement types per pair (`SYNAPSE_COUNT`, `SUM_ANATOMICAL_SIZE`).

In [25]:
conn_ex1 = conn_df.filter(
    pl.col("pre_nuc_id_str").is_in(proofread_nuc_ids)
    & pl.col("post_nuc_id_str").is_in(csm_nuc_ids)
)
print(f"Example 1 filtered rows ((proofread ∩ CSM)-pre × CSM-post): {conn_ex1.shape[0]}")

Example 1 filtered rows ((proofread ∩ CSM)-pre × CSM-post): 583750


In [26]:
pre_ids_ex1 = set(conn_ex1["pre_nuc_id_str"].to_list())
post_ids_ex1 = set(conn_ex1["post_nuc_id_str"].to_list())
missing_ex1 = (pre_ids_ex1 | post_ids_ex1) - existing_items
assert len(missing_ex1) == 0, (
    f"{len(missing_ex1)} cell ids in Example 1 not found in dataitem/."
)
print(f"All {len(pre_ids_ex1 | post_ids_ex1)} unique cell ids confirmed in dataitem/.")

All 34313 unique cell ids confirmed in dataitem/.


In [27]:
rows_ex1 = []
for row in conn_ex1.iter_rows(named=True):
    pre = row["pre_nuc_id_str"]
    post = row["post_nuc_id_str"]
    rows_ex1.append(
        CellCellConnectivityLong(
            id=f"{pre}_{post}_{SynapticMeasurementType.SYNAPSE_COUNT.value}",
            presynaptic_cell=pre,
            postsynaptic_cell=post,
            measurement_type=SynapticMeasurementType.SYNAPSE_COUNT.value,
            modality=Modality.ELECTRON_MICROSCOPY.value,
            value=float(row["n_syn"]),
            unit=Unit.COUNT.value,
            project_id=PROJECT_ID,
        )
    )
    rows_ex1.append(
        CellCellConnectivityLong(
            id=f"{pre}_{post}_{SynapticMeasurementType.SUM_ANATOMICAL_SIZE.value}",
            presynaptic_cell=pre,
            postsynaptic_cell=post,
            measurement_type=SynapticMeasurementType.SUM_ANATOMICAL_SIZE.value,
            modality=Modality.ELECTRON_MICROSCOPY.value,
            value=float(row["sum_size"]),
            unit=Unit.ARBITRARY_UNIT.value,
            project_id=PROJECT_ID,
        )
    )

print(f"CellCellConnectivityLong rows (Example 1): {len(rows_ex1)}")

CellCellConnectivityLong rows (Example 1): 1167500


In [28]:
schema_cc = build_arrow_schema(CellCellConnectivityLong)
table_ex1 = attach_linkml_metadata(
    models_to_table(rows_ex1, schema=schema_cc),
    linkml_class="CellCellConnectivityLong",
)

write_deltalake(
    OUTPUT_ROOT + "cellcellconnectivitylong_proofread_pre_to_csm_post/",
    table_ex1,
    mode="overwrite",
    predicate=f"project_id = '{PROJECT_ID}'",
    partition_by=["project_id", "measurement_type"],
)
print(f"Written to cellcellconnectivitylong_proofread_pre_to_csm_post/: {table_ex1.num_rows} rows")

Written to cellcellconnectivitylong_proofread_pre_to_csm_post/: 1167500 rows


In [29]:
ex1_v = pl.read_delta(OUTPUT_ROOT + "cellcellconnectivitylong_proofread_pre_to_csm_post/").filter(
    pl.col("project_id") == PROJECT_ID
)
print(f"Shape: {ex1_v.shape}")
print(ex1_v.group_by("measurement_type").len())
print(ex1_v.head(3))
assert ex1_v.shape[0] == len(rows_ex1)
assert ex1_v["measurement_type"].n_unique() == 2, "Expected both SYNAPSE_COUNT and SUM_ANATOMICAL_SIZE"

Shape: (1167500, 9)
shape: (2, 2)
┌─────────────────────┬────────┐
│ measurement_type    ┆ len    │
│ ---                 ┆ ---    │
│ str                 ┆ u32    │
╞═════════════════════╪════════╡
│ SYNAPSE_COUNT       ┆ 583750 │
│ SUM_ANATOMICAL_SIZE ┆ 583750 │
└─────────────────────┴────────┘
shape: (3, 9)
┌─────────────┬────────────┬────────────┬────────────┬───┬───────┬───────┬────────────┬────────────┐
│ id          ┆ descriptio ┆ presynapti ┆ postsynapt ┆ … ┆ value ┆ unit  ┆ project_id ┆ measuremen │
│ ---         ┆ n          ┆ c_cell     ┆ ic_cell    ┆   ┆ ---   ┆ ---   ┆ ---        ┆ t_type     │
│ str         ┆ ---        ┆ ---        ┆ ---        ┆   ┆ f64   ┆ str   ┆ str        ┆ ---        │
│             ┆ str        ┆ str        ┆ str        ┆   ┆       ┆       ┆            ┆ str        │
╞═════════════╪════════════╪════════════╪════════════╪═══╪═══════╪═══════╪════════════╪════════════╡
│ 301245_3614 ┆ null       ┆ 301245     ┆ 361468     ┆ … ┆ 1.0   ┆ COUNT ┆ minnie6

---
## Example 2 — (proofread ∩ CSM)-pre × (proofread ∩ CSM)-post

Filter the same parquet to pairs where both pre and post are in the proofread ∩ CSM set. Write `SYNAPSE_COUNT` only.

In [30]:
conn_ex2 = conn_df.filter(
    pl.col("pre_nuc_id_str").is_in(proofread_nuc_ids)
    & pl.col("post_nuc_id_str").is_in(proofread_nuc_ids)
)
print(f"Example 2 filtered rows ((proofread ∩ CSM) × (proofread ∩ CSM)): {conn_ex2.shape[0]}")

Example 2 filtered rows ((proofread ∩ CSM) × (proofread ∩ CSM)): 83877


In [31]:
pre_ids_ex2 = set(conn_ex2["pre_nuc_id_str"].to_list())
post_ids_ex2 = set(conn_ex2["post_nuc_id_str"].to_list())
missing_ex2 = (pre_ids_ex2 | post_ids_ex2) - existing_items
assert len(missing_ex2) == 0, (
    f"{len(missing_ex2)} cell ids in Example 2 not found in dataitem/."
)
print(f"All {len(pre_ids_ex2 | post_ids_ex2)} unique cell ids confirmed in dataitem/.")

All 1636 unique cell ids confirmed in dataitem/.


In [32]:
rows_ex2 = []
for row in conn_ex2.iter_rows(named=True):
    pre = row["pre_nuc_id_str"]
    post = row["post_nuc_id_str"]
    rows_ex2.append(
        CellCellConnectivityLong(
            id=f"{pre}_{post}_{SynapticMeasurementType.SYNAPSE_COUNT.value}",
            presynaptic_cell=pre,
            postsynaptic_cell=post,
            measurement_type=SynapticMeasurementType.SYNAPSE_COUNT.value,
            modality=Modality.ELECTRON_MICROSCOPY.value,
            value=float(row["n_syn"]),
            unit=Unit.COUNT.value,
            project_id=PROJECT_ID,
        )
    )

print(f"CellCellConnectivityLong rows (Example 2): {len(rows_ex2)}")

CellCellConnectivityLong rows (Example 2): 83877


In [33]:
table_ex2 = attach_linkml_metadata(
    models_to_table(rows_ex2, schema=schema_cc),
    linkml_class="CellCellConnectivityLong",
)

write_deltalake(
    OUTPUT_ROOT + "cellcellconnectivitylong_proofread_to_proofread/",
    table_ex2,
    mode="overwrite",
    predicate=f"project_id = '{PROJECT_ID}'",
    partition_by=["project_id", "measurement_type"],
)
print(f"Written to cellcellconnectivitylong_proofread_to_proofread/: {table_ex2.num_rows} rows")

Written to cellcellconnectivitylong_proofread_to_proofread/: 83877 rows


In [34]:
ex2_v = pl.read_delta(
    OUTPUT_ROOT + "cellcellconnectivitylong_proofread_to_proofread/"
).filter(pl.col("project_id") == PROJECT_ID)
print(f"Shape: {ex2_v.shape}")
print(ex2_v.group_by("measurement_type").len())
print(ex2_v.head(3))
assert ex2_v.shape[0] == len(rows_ex2)
assert ex2_v["measurement_type"].n_unique() == 1
assert ex2_v["measurement_type"][0] == SynapticMeasurementType.SYNAPSE_COUNT.value

Shape: (83877, 9)
shape: (1, 2)
┌──────────────────┬───────┐
│ measurement_type ┆ len   │
│ ---              ┆ ---   │
│ str              ┆ u32   │
╞══════════════════╪═══════╡
│ SYNAPSE_COUNT    ┆ 83877 │
└──────────────────┴───────┘
shape: (3, 9)
┌─────────────┬────────────┬────────────┬────────────┬───┬───────┬───────┬────────────┬────────────┐
│ id          ┆ descriptio ┆ presynapti ┆ postsynapt ┆ … ┆ value ┆ unit  ┆ project_id ┆ measuremen │
│ ---         ┆ n          ┆ c_cell     ┆ ic_cell    ┆   ┆ ---   ┆ ---   ┆ ---        ┆ t_type     │
│ str         ┆ ---        ┆ ---        ┆ ---        ┆   ┆ f64   ┆ str   ┆ str        ┆ ---        │
│             ┆ str        ┆ str        ┆ str        ┆   ┆       ┆       ┆            ┆ str        │
╞═════════════╪════════════╪════════════╪════════════╪═══╪═══════╪═══════╪════════════╪════════════╡
│ 226128_5188 ┆ null       ┆ 226128     ┆ 518848     ┆ … ┆ 1.0   ┆ COUNT ┆ minnie65   ┆ SYNAPSE_CO │
│ 48_SYNAPSE_ ┆            ┆            ┆   

---
## Summary

| Path | Rows |
|------|------|
| `dataset/` | +1 (`minnie65_v1300_proofread` = proofread ∩ CSM cells) |
| `dataitem_dataset_association/` | one per proofread ∩ CSM cell |
| `cellcellconnectivitylong_proofread_pre_to_csm_post/` | 2 × filtered pairs: (proofread ∩ CSM)-pre × CSM-post (`SYNAPSE_COUNT` + `SUM_ANATOMICAL_SIZE`) |
| `cellcellconnectivitylong_proofread_to_proofread/` | 1 × filtered pairs: (proofread ∩ CSM) × (proofread ∩ CSM) (`SYNAPSE_COUNT` only) |

Both examples use the same precomputed `minnie_soma_soma_connectivity.parquet` with different filters.